# Proyecto de aproximación de Apophis en abril de 2029
## Lizeth Serna Santa
### Curso de Mecánica Celeste - Universidad de Antioquia

En este proyecto se aplicarán los conceptos aprendidos en el curso de mecánica celeste alrededor de las temáticas de integración de N-cuerpos y de dos cuerpos.

In [1]:
%pip install pymcel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
# Load the data

pymcel version  0.6.31


In [7]:
help(pc.consulta_horizons)

Help on function consulta_horizons in module pymcel:

consulta_horizons(
    id='399',
    location='@0',
    epochs=None,
    datos='vectors',
    propiedades='default'
)
    Realiza una consulta en Horizons usando astroquery.

    Opciones:
        id, location, epochs: entradas comunes de Horizons.
        Se puede pasar una epoca como una única fecha o una lista de fechas.

            Si se piden elementos o vectores 'epochs' son fechas en TDB (tiempo dinámico del baricentro).
            Si se piden efemérides 'epochs' son fechas en UTC.

        datos: datos requeridos. Valores aceptados: 'vectors', 'elements', 'ephemeris'

        propiedades: lista con las propiedades que se quieren extraer.

            Ejemplo: propiedades = propiedades = [('x','m'),('y','m'),('z','m')]

            Si propiedades se pasa como 'default' se extraen las propiedades por defecto
            de acuerdo con 'datos' así:
                'vectors': vector de estado en SI.
                'elements':

Se utiliza `consulta_horizons` de `pymcel` para obtener los parametros fisicos de la Tierra, el Sol y Apophis.
Si Horizons solo entrega el parametro gravitacional $GM$, la masa se calcula con:
$$
M = \frac{GM}{G}
$$
con $G = 6.67430\times 10^{-11}\ \mathrm{m^3\,kg^{-1}\,s^{-2}}$.
Las masas finales se expresan en unidades de masa solar:
$$
M_{\odot} = \frac{M_{\mathrm{kg}}}{M_{\odot,\mathrm{kg}}}
$$
Se verifica la unidad de $GM$ (si esta en $\mathrm{km^3\,s^{-2}}$, se convierte a $\mathrm{m^3\,s^{-2}}$ con un factor $10^9$).

Si la masa no esta disponible en Horizons, se estima con volumen esferico:
$$
M = \frac{4}{3}\pi R^3 \rho
$$
Usamos el diametro $D = 0.34\ \mathrm{km}$ de SBDB (JPL) para Apophis y la densidad de referencia $\rho = 1.5\ \mathrm{g\,cm^{-3}}$ empleada para Apophis en Taylor et al. (2023, PSJ 4 79).
Fuentes:
- JPL SBDB: https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=99942
- Taylor et al. 2023 (PSJ, DOI: 10.3847/PSJ/acccef)

In [ ]:
# Consulta de masas con pymcel
# Valores fisicos en SI
import pymcel.constantes as const

G_SI = const.G  # m^3 kg^-1 s^-2
M_SUN_KG = const.M_sun  # kg

# Horizons requiere una epoca valida
epochs = "2029-04-13 00:00:00"  # UTC para efemerides

def _scalar_from_value(value):
    if hasattr(value, "iloc"):
        return float(value.iloc[0])
    if isinstance(value, (list, tuple, np.ndarray)):
        return float(value[0])
    return float(value)

def _extract_mass_kg(result):
    # Si llega el resultado completo de consulta_horizons
    if isinstance(result, tuple) and len(result) == 3:
        result = result[2]

    # DataFrame de pandas
    if hasattr(result, "columns"):
        for key in ("MASS", "mass", "Mass", "M", "GM", "gm", "GMS", "GM_SI"):
            if key in result.columns:
                value = _scalar_from_value(result[key])
                if key.upper().startswith("GM"):
                    value = value * 1.0e9  # (km^3/s^2) -> (m^3/s^2)
                    return value / G_SI
                return value

    # Estructuras tipo dict
    for key in ("MASS", "mass", "Mass", "M"):
        if hasattr(result, "__contains__") and key in result:
            return _scalar_from_value(result[key])
    for key in ("GM", "gm", "GMS", "GM_SI"):
        if hasattr(result, "__contains__") and key in result:
            gm_value = _scalar_from_value(result[key])
            gm_value_m3_s2 = gm_value * 1.0e9  # (km^3/s^2) -> (m^3/s^2)
            return gm_value_m3_s2 / G_SI
    return None

def _describe_result(label, result):
    print(f"\n{label} -> type: {type(result)}")
    if hasattr(result, "columns"):
        print("columns:", list(result.columns))
    elif hasattr(result, "keys"):
        try:
            print("keys:", list(result.keys()))
        except Exception:
            pass

def _require_mass(label, mass_kg):
    if mass_kg is None:
        print(f"Advertencia: no se encontro masa/GM para {label}.")
        return np.nan
    return mass_kg

# Masas de Sol y Tierra desde constantes de pymcel (SI)
M_sun_kg = const.M_sun  # kg
M_earth_kg = const.M_earth  # kg

# Apophis: intentar Horizons
try:
    tabla_ap, ts_ap, salida_ap = pc.consulta_horizons(
        "99942",
        epochs=epochs,
        datos="ephemeris",
        propiedades="default",
    )
    _describe_result("Apophis", salida_ap)
    M_apophis_kg = _require_mass("Apophis", _extract_mass_kg(salida_ap))  # kg
except Exception as exc:
    print(f"Error al consultar Horizons para Apophis: {exc}")
    M_apophis_kg = np.nan

# Fallback: masa estimada con diametro y densidad de referencia
if np.isnan(M_apophis_kg):
    D_APOPHIS_KM = 0.34  # km (JPL SBDB)
    R_APOPHIS_M = D_APOPHIS_KM * 1.0e3 / 2.0  # m
    RHO_APOPHIS = 1.5e3  # kg/m^3 (Taylor et al. 2023, PSJ 4 79)
    M_apophis_kg = (4.0 / 3.0) * np.pi * (R_APOPHIS_M ** 3) * RHO_APOPHIS  # kg
    print("Usando masa estimada para Apophis (diametro + densidad de referencia).")

# Conversion a unidades de masa solar (Msun)
M_sun = M_sun_kg / M_SUN_KG  # Msun
M_earth = M_earth_kg / M_SUN_KG  # Msun
M_apophis = M_apophis_kg / M_SUN_KG  # Msun

print("Masas en Msun:")
print(f"  Sol: {M_sun:.6e} Msun")
print(f"  Tierra: {M_earth:.6e} Msun")
print(f"  Apophis: {M_apophis:.6e} Msun")


Error al consultar Horizons para Apophis: 'HorizonsClass' object has no attribute 'ephemeris'
Masas en Msun:
  Sol: 1.000000e+00 Msun
  Tierra: 3.003489e-06 Msun
  Apophis: nan Msun


c:\Users\Lizeth\Downloads\Anaconda\Lib\site-packages\erfa\core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
